# QKV Attention Through a Different Lens

The famous [Attention Is All You Need paper](https://arxiv.org/abs/1706.03762) introduced the Transformer architecture, a neural network based on the query-key-value attention mechanism, for natural language translation. These models later became known to be highly effective in several other machine learning fields (including other [language processing](https://en.wikipedia.org/wiki/Natural_language_processing) tasks, [computer vision](https://en.wikipedia.org/wiki/Vision_transformer), [music source separation](https://arxiv.org/abs/2211.08553), etc.), so it is worth sudying their operation from a more general perspective than language processing or sequence modeling.

## Notation

We are going to use row-major notation: $xA = b \in \mathbb{R}^m$ for $m, n \in \mathbb{N}$ and $A \in \mathbb{R}^{n \times m}$ and $x \in \mathbb{R}^{1 \times n}$ (or $x \in \mathbb{R}^n$ for short).

## Feed-forward neural network

For a model dimensionality $d_m \in \mathbb{N}$, let $x \in \mathbb{R}^{d_m}$ be an input vector. A two-layered [feed-forward neural network](https://en.wikipedia.org/wiki/Feedforward_neural_network) computes the following expression:

$$
y = f^{(1)}\left( x \cdot W^{(1)} + b^{(1)} \right) \cdot W^{(2)}
    + b^{(2)} \quad \in \mathbb{R}^{d^{(2)}}
$$

where, for some $d^{(1)},\ d^{(2)} \in \mathbb{N}$:

 * $W^{(1)} \in \mathbb{R}^{d_m \times d^{(1)}}$ and $W^{(2)} \in \mathbb{R}^{d^{(1)} \times d^{(2)}}$ are weight matrices,

 * $b^{(1)} \in \mathbb{R}^{d^{(1)}}$ and $b^{(2)} \in \mathbb{R}^{d^{(2)}}$ are bias vectors,

 * $f^{(1)} : \mathbb{R}^{d^{(1)}} \rightarrow \mathbb{R}^{d^{(1)}}$ is some non-linear function, e.g. $\text{ReLU}(x) = \max(0, x)$.

### Parallel version

For some $d_s \in \mathbb{N}$ let's replace $x$ by $X \in \mathbb{R}^{d_s \times d_m}$ and broadcast $b^{(k)}$ into $d_s$ dimensions in order to evaluate the network for multiple input samples simultaneously (with $d_m$ features and $d_s$ samples):

$$
Y = f^{(1)}\left( X \cdot W^{(1)} + B^{(1)} \right) \cdot W^{(2)}
      + B^{(2)} \quad \in \mathbb{R}^{d_s \times d^{(2)}}
$$

where $B^{(k)} = \left[ b^{(k)} \right]_{i=1}^{d_s} \in \mathbb{R}^{d_s \times d^{(k)}}$.

Note: treating $X$ as a $d_s$ long sequence instead of an unordered collection and using different, position-dependent vectors in $B^{(1)}$ is analogous to [positional encoding in Transformers](https://en.wikipedia.org/wiki/Transformer_%28deep_learning%29#Positional_encoding).

### Universal approximation theorem

The gist of it is that the power of neural networks comes from the fact that [any continuous function can be approximated well by some neural network](https://en.wikipedia.org/wiki/Universal_approximation_theorem) - and many real-world problems can be mapped to continuous functions, i.e. functions where a small change in the argument leads to a small change in the function's value.

## QKV-Attention

The heart and soul of the Transformer (besides [residual connections](https://en.wikipedia.org/wiki/Residual_neural_network) and a few other tricks) is the multi-headed scaled dot-product attention, with optional masking (depending on the problem).

A single attention head calculates the following expression (row-major notation):

For some $d_m,\ d_s \in \mathbb{N}$ and $X^Q,\ X^K,\ X^V \in \mathbb{R}^{d_s \times d_m}$ input matrices:

$$
\begin{align*}
  Y = & \text{Attention}(Q, K, V) \\
    = & \text{softmax}\left( \frac{Q \cdot K^T}{\sqrt{d^K}} + M \right)
        \cdot V
    \quad \in \mathbb{R}^{d_s \times d^V}
\end{align*}
$$

where:

 * $Q = X^Q \cdot W^Q$, $K = X^K \cdot W^K$, and $V = X^V \cdot W^V$,

 * $W^Q,\ W^K \in \mathbb{R}^{d_m \times d^K}$ and $W^V \in \mathbb{R}^{d_m \times d^V}$ are learnable weights for some $d_m \ge d^K,\ d^V \in \mathbb{N}$,

 * $\text{softmax}$ is taken row-wise independently,

 * and $M \in \mathbb{R}^{d_s \times d_s}$ is a constant called the "_mask_" and is used for suppressing certain attention weights during training autoregressive models, otherwise it can be omitted by setting $M = 0 \in \mathbb{R}^{d_s \times d_s}$.

The resulting $Y$ vectors from the individual attention heads are then concatenated to form higher dimensional vectors which then can be used in residual connections.

### Variants

#### Self-attention (transformer encoder)

For some $X \in \mathbb{R}^{d_s \times d_m}$, let $X^Q = X^K = X^V = X$.

#### Cross-attention (transformer decoder)

For some $X_1, X_2 \in \mathbb{R}^{d_s \times d_m}$ pair of inputs, let $X^Q = X_1$ and let $X^K = X^V = X_2$.

## Rearranging, reinterpreting

After substitutions, and utilizing the associativity of matrix multiplication and the $(A \cdot B)^T = B^T \cdot A^T$ identity, we get:

$$
\begin{align*}
Y & = \text{Attention}\left( X^Q \cdot W^Q, \quad X^K \cdot W^K, \quad X^V \cdot W^V \right) & \\
  & = \text{softmax}\left(
        \frac{
          \left( X^Q \cdot W^Q \right) \cdot \left( X^K \cdot W^K \right)^T
        }{\sqrt{d^K}} + M \right)
        \cdot X^V \cdot W^V
      & \in \mathbb{R}^{d_s \times d^V} \\
  & = \text{softmax} \left(
            X^Q
            \cdot \left( \frac{1}{\sqrt{d^K}} \cdot W^Q \cdot \left( W^K \right)^T \right)
            \cdot \left( X^K \right)^T
          + M \right)
        \cdot X^V \cdot W^V
      &
\end{align*}
$$

Let:

$$
\begin{align*}
  W^{(1)} & = \frac{1}{\sqrt{d^K}} \cdot W^Q \cdot \left( W^K \right)^T & \in \mathbb{R}^{d_m \times d_m} \\
  W^{(2)} & = W^V & \in \mathbb{R}^{d_m \times d^V} \\
  W_{X^K}^{(1)} & = W^{(1)} \cdot \left( X^K \right)^T & \in \mathbb{R}^{d_m \times d_s} \\
  W_{X^V}^{(2)} & = X^V \cdot W^{(2)} & \in \mathbb{R}^{d_s\times d^V} \\
  B^{(1)} & = M & \in \mathbb{R}^{d_s \times d_s}\\
  B^{(2)} & = 0 & \in \mathbb{R}^{d_s \times d^V} \\
  f^{(1)} & = \text{softmax} &
\end{align*}
$$

Then the attention mechanism becomes:

$$
Y = f^{(1)} \left( X^Q \cdot W_{X^K}^{(1)} + B^{(1)} \right) \cdot W_{X^V}^{(2)} + B^{(2)}
$$

Furthermore:

 * Self-attention for $X \in \mathbb{R}^{d_s \times d_m}$ can also be written as:

   $$
   Y = f^{(1)} \left( X \cdot W^{(1)} \cdot X^T + B^{(1)} \right)
       \cdot \left( X \cdot W^{(2)} \right) + B^{(2)}
   $$

 * Cross-attention for $X_1, X_2 \in \mathbb{R}^{d_s \times d_m}$ becomes:

   $$
   Y = f^{(1)} \left( X_1 \cdot W^{(1)} \cdot X_2^T + B^{(1)} \right)
       \cdot \left( X_2 \cdot W^{(2)} \right) + B^{(2)}
   $$

### Notes

 * The $\text{softmax}$ function can be thought of as a smooth and normalized version of the $\max(x, 0)$ function, i.e. a smooth and normalized $\text{ReLU}$.

    * According to the [literature](https://arxiv.org/abs/2309.08586), in some circumstances, it may even be beneficial to replace $\text{softmax}$ with scaled $\text{ReLU}$, since it can help with parallelization while achieving similar performance e.g. in [Vision Transformers](https://en.wikipedia.org/wiki/Vision_transformer).

 * $W^{(1)}$ can be thought of as a low-rank decomposition of some $W_*^{(1)}$ weight matrix.

 * $W_{X^K}^{(1)}$ and $W_{X^V}^{(2)}$ can be thought of as weight matrices generated from the input on-the-fly for a two-layered feed-forward neural network with $B^{(1)}$ and $B^{(2)}$ as bias.

 * The self-attention formula contains a [quadratic form](https://en.wikipedia.org/wiki/Quadratic_form), analogous to evaluating the first layer for a second degree polynomial instead of the raw input features. Then the resulting activations are multiplied again by $X$ in the second layer, as if introducing a third degree (aside from the non-linearity). (Cf. [kernel trick](https://en.wikipedia.org/wiki/Kernel_method), [polynomial regression](https://en.wikipedia.org/wiki/Polynomial_regression)).

 * Similarly, cross-attention can be thought of as a network which operates on the product of its two inputs.

 * If the linear projections of the input are viewed as input-dependent weight matrices, then QKV-attention becomes a [hypernetwork](https://doi.org/10.48550/arXiv.2406.05816) (also known as [fast weight programmer](https://doi.org/10.48550/arXiv.2102.11174)).